# Fase 1: primeira tabela Delta no MinIO

Objetivo: provar que as três peças conversam.

- **Spark** processa
- **Delta Lake** define o formato da tabela
- **MinIO** guarda os arquivos

Rode uma célula por vez (Shift+Enter) e leia a saída antes de seguir.

## 1. Abrir a sessão Spark

A função vem do nosso pacote `gabs`. Ela lê o endereço e as credenciais do MinIO das variáveis de ambiente do contêiner.

In [ ]:
from gabs import storage
from gabs.spark import get_spark_session

spark = get_spark_session("fase1")
print("Spark", spark.version)

## 2. Gravar uma tabela Delta na camada bronze

In [ ]:
caminho = storage.path("bronze", "exemplo", "cidades")
print(caminho)

dados = [
    (1, "Natal", "RN"),
    (2, "Parnamirim", "RN"),
    (3, "Fortaleza", "CE"),
]
df = spark.createDataFrame(dados, ["ID_CIDADE", "NM_CIDADE", "SG_UF"])

df.write.format("delta").mode("overwrite").save(caminho)

Agora abra o console do MinIO (porta 9001), entre no bucket `bronze` e navegue até `exemplo/cidades`.

Você verá arquivos `.parquet` (os dados) e uma pasta `_delta_log` (o diário de transações). **Delta é isto: Parquet mais um log.**

## 3. Ler de volta

In [ ]:
spark.read.format("delta").load(caminho).orderBy("ID_CIDADE").show()

## 4. Alterar a tabela

Um `append` e um `update`. Cada operação vira uma nova versão no log.

In [ ]:
from delta.tables import DeltaTable

novas = spark.createDataFrame([(4, "Mossoro", "RN")], ["ID_CIDADE", "NM_CIDADE", "SG_UF"])
novas.write.format("delta").mode("append").save(caminho)

tabela = DeltaTable.forPath(spark, caminho)
tabela.update(condition="ID_CIDADE = 4", set={"NM_CIDADE": "'Mossoró'"})

tabela.toDF().orderBy("ID_CIDADE").show()

## 5. Histórico

O que o Parquet puro não tem: quem mudou o quê, e quando.

In [ ]:
tabela.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## 6. Time travel

Ler a tabela como ela era na versão 0, antes do append e do update.

In [ ]:
spark.read.format("delta").option("versionAsOf", 0).load(caminho).orderBy("ID_CIDADE").show()

## 7. Encerrar

Libera a memória do Spark. Enquanto a sessão está aberta, a Spark UI responde na porta 4040.

In [ ]:
spark.stop()